## 1. Imports and reproducibility
NumPy creates the complex channel and symbols. TensorFlow defines and trains the unfolded detector. Matplotlib displays training loss and BER. A fixed seed makes runs easier to compare; numerical differences between devices can remain.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)
plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 11})

## 2. Experiment settings
`Nt` is the number of transmitted streams and `Nr` the satellite receive antennas. Each channel realization carries `Num_ii` independent symbol vectors at each SNR. With the defaults, there are 6,000 samples: six SNRs, ten channels, and 100 vectors per channel/SNR.

The experiment uses normalized 16-QAM: each real component is in `[-3,-1,1,3]/sqrt(10)` and each complex symbol has unit average energy. Fifteen unfolded layers learn three scalars each (45 trainable scalars). The settings retain the uploaded experiment's learning rate, layer count, channel parameters, and alternating sample split.

In [ ]:

SNR_DB = np.arange(10, 22, 2)
Nt, Nr = 8, 16
QAM = 16
ENERGY = 2 * (QAM - 1) / 3
N_LAYERS = 15
TRAIN_STEPS = int(os.environ.get("TORNET_STEPS", "2000"))
LEARNING_RATE = 0.01
BAND_WIDTH = max(1, (Nt - 1) // 2)
assert TRAIN_STEPS > 0
print(f"Layers: {N_LAYERS}; training steps: {TRAIN_STEPS}; QAM: {QAM}")

## 3. Direct user-to-satellite channel
Generate one complex matrix $H_d$ of shape `Nr x Nt` per channel realization. Each entry combines a diffuse Gaussian component and a random-phase line-of-sight component. A common lognormal amplitude and Gaussian phase impairment multiply the entire matrix:
$$H_d[i,j]=a e^{j\delta}\left(g_{ij}+\sqrt{v_{LoS,ij}}e^{j\phi_{ij}}\right),$$
where $a\sim\mathrm{Lognormal}(\mu_r,\sigma_r)$, $\delta\sim\mathcal N(0,\sigma_\delta^2)$, $g_{ij}\sim\mathcal{CN}(0,v_{ray,ij})$, and $\phi_{ij}\sim U[-\pi,\pi]$. The diffuse and LoS powers are drawn from their configured ranges.

Normalize the channel by `sqrt(Nr)` and use only the direct propagation model:
$$H=H_d/\sqrt{N_r},\qquad y=Hs+n.$$
The direct-link fading distribution and normalization match the previous experiment (its effective lognormal scale factor was one). There is no reflected term or phase-control matrix. One channel is fixed over all symbol vectors and SNRs for that realization. Noise has complex variance $10^{-\mathrm{SNR}/10}$. Optional channel-estimation error affects the detector's channel estimate, not the physical received signal; it defaults to zero.

## 4. Generate symbols, observations, and channel matrices
Each returned dictionary is keyed by SNR. Within one SNR, samples are ordered by channel realization and then symbol vector. `s` contains transmitted vectors, `H` the detector's channel estimate, `r` the received vectors, and `noise_by_snr` the simulated noise. The noise is retained here for the original regularization calculation.

In [ ]:

s, H, r, noise_by_snr = Load_data()
for snr in SNR_DB:
    np.testing.assert_allclose(r[snr], H[snr] @ s[snr] + noise_by_snr[snr], atol=1e-10)
for i, cc in enumerate(Num_cc):
    np.testing.assert_allclose(H[SNR_DB[0]][i * Num_ii], data.H_direct[cc])
print("Per-SNR shapes:", s[SNR_DB[0]].shape, H[SNR_DB[0]].shape,
      r[SNR_DB[0]].shape)

## 5. Convert the complex system to a real system
The network uses real-valued tensors. Stack real and imaginary symbol/receive components, and use the block matrix
$$H_R=\begin{bmatrix}\Re(H)&-\Im(H)\\\Im(H)&\Re(H)\end{bmatrix}.$$
Then $y_R=H_Rs_R+n_R$, with `2*Nt` unknown real components and `2*Nr` observations. Concatenating SNR blocks keeps their order; `snr_labels` records the SNR of every sample for later evaluation.

In [ ]:
def real_vector(z):
    return np.concatenate([z.real, z.imag], axis=1)

def real_channel(z):
    top = np.concatenate([z.real, -z.imag], axis=2)
    bottom = np.concatenate([z.imag, z.real], axis=2)
    return np.concatenate([top, bottom], axis=1)

s_real = np.concatenate([real_vector(s[v]) for v in SNR_DB])
H_real = np.concatenate([real_channel(H[v]) for v in SNR_DB])
y_real = np.concatenate([real_vector(r[v]) for v in SNR_DB])
noise_real = np.concatenate([real_vector(noise_by_snr[v]) for v in SNR_DB])
snr_labels = np.concatenate([np.full(len(s[v]), v) for v in SNR_DB])
np.testing.assert_allclose(y_real, H_real @ s_real + noise_real, atol=1e-10)
print("Real symbols:", s_real.shape, "; real channel:", H_real.shape)

## 6. Build the regularized system and split its lower bands
Construct $A=H_R^TH_R+\lambda I$ and $b=H_R^Ty_R$. TORNET uses these matrices as structured inputs; this step does not run a separate detector.

For compatibility, $\lambda$ is the sample variance of each simulated real noise vector, as in the uploaded code. This is a simulation-only choice because the actual noise vector is known here; a practical receiver should use an estimated noise variance. With unit complex symbol energy, the nominal alternative in this real representation is `10**(-snr_labels/10)`.

Split $A=D+L+U$ into diagonal, strictly lower, and strictly upper parts. Split $L=E_{band}+F$, where `E_band` contains the first `BAND_WIDTH` lower diagonals. `ENERGY` names QAM normalization separately to avoid the source notebook's ambiguous `E`. The band width retains the original choice based on `Nt`.

In [ ]:
regularizer = np.var(noise_real, axis=1, keepdims=True)
Ht = np.transpose(H_real, (0, 2, 1))
A = Ht @ H_real + regularizer * np.eye(2 * Nt)
b = Ht @ y_real
L = np.tril(A, -1)
U = np.triu(A, 1)
D = A - L - U
E_band = np.tril(L, -1) - np.tril(L, -(BAND_WIDTH + 1))
F = L - E_band
np.testing.assert_allclose(A, D + E_band + F + U)
print("A:", A.shape, "; b:", b.shape, "; lower-band width:", BAND_WIDTH)

## 7. Training and test tensors
Even-indexed samples train the network; odd-indexed samples test it. Both sets therefore include every SNR and share channel realizations, but use different symbol/noise draws. This measures performance on new symbols under seen channel realizations, not generalization to unseen channels.

All network inputs are explicitly converted to float32. The initial estimate is zero. Each input list has the order `[x0, D, E_band, F, U, b]`. Ground-truth symbols are used only by the training loss and BER calculation.

In [ ]:
train_idx = np.arange(0, len(s_real), 2)
test_idx = np.arange(1, len(s_real), 2)

def model_inputs(indices):
    return [tf.convert_to_tensor(v, dtype=tf.float32) for v in
            [np.zeros_like(s_real[indices]), D[indices], E_band[indices],
             F[indices], U[indices], b[indices]]]

train_inputs = model_inputs(train_idx)
test_inputs = model_inputs(test_idx)
train_targets = tf.convert_to_tensor(s_real[train_idx], dtype=tf.float32)
test_targets = s_real[test_idx]
print("Training samples:", len(train_idx), "; test samples:", len(test_idx))

## 8. Constellation-aware activation
For normalized 16-QAM, multiply by $\sqrt{10}$, apply a sum of three tanh functions centered at -2, 0, and 2, and divide by $\sqrt{10}$. The fixed slope 1.5 follows the original code. This smooth mapping encourages outputs near the four real constellation levels and remains differentiable during training. Final BER uses hard nearest-level decisions.

In [ ]:
def constellation_activation(x):
    z = tf.sqrt(tf.cast(ENERGY, x.dtype)) * x
    slope = tf.cast(1.5, x.dtype)
    return (tf.tanh(slope * z) + tf.tanh(slope * (z - 2))
            + tf.tanh(slope * (z + 2))) / tf.sqrt(tf.cast(ENERGY, x.dtype))

## 9. One unfolded TORNET layer
Each layer learns three independent real scalars, initialized to `alpha=0.6`, `beta=0.4`, `gamma=0.85`. Define
$$M=D+\alpha E_{band}+\beta F,$$
$$q=[(1-\gamma)D-(\gamma-\alpha)E_{band}-(\gamma-\beta)F-\gamma U]x+\gamma b.$$
Solve $M\tilde{x}=q$ and apply the constellation activation. Because $M$ is lower triangular, `tf.linalg.triangular_solve` implements forward substitution without explicitly forming an inverse. This is mathematically equivalent to the uploaded update and matches the paper's triangular implementation.

The scalars are unconstrained trainable parameters, as in the supplied model. Three weights per layer are shared across all samples; channel dependence comes from the input matrices.

In [ ]:
class TORNETLayer(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        for name, initial in [("alpha", 0.6), ("beta", 0.4), ("gamma", 0.85)]:
            setattr(self, name, self.add_weight(
                name=name, shape=(), initializer=keras.initializers.Constant(initial),
                trainable=True, dtype="float32"))

    def call(self, inputs):
        x, d, e, f, u, rhs_b = inputs
        rhs = ((1 - self.gamma) * d - (self.gamma - self.alpha) * e
               - (self.gamma - self.beta) * f - self.gamma * u) @ x
        rhs = rhs + self.gamma * rhs_b
        lower = d + self.alpha * e + self.beta * f
        estimate = tf.linalg.triangular_solve(lower, rhs, lower=True)
        return constellation_activation(estimate)

## 10. Stack the layers into TORNET
The first layer starts from zero. Each subsequent layer receives the previous estimate and the same system matrices. The model returns both the concatenated intermediate estimates (`batch x 2*Nt x layers`) and the final estimate (`batch x 2*Nt x 1`). Intermediate estimates supervise every layer during training; only the last estimate is demodulated for BER.

In [ ]:
class TORNET(keras.Model):
    def __init__(self, depth):
        super().__init__(name="TORNET")
        self.tor_layers = [TORNETLayer(name=f"tor_layer_{k+1}") for k in range(depth)]

    def call(self, inputs):
        x, d, e, f, u, rhs_b = inputs
        estimates = []
        for layer in self.tor_layers:
            x = layer([x, d, e, f, u, rhs_b])
            estimates.append(x)
        return tf.concat(estimates, axis=2), x

model = TORNET(N_LAYERS)
_ = model([v[:1] for v in train_inputs])
assert model.count_params() == 3 * N_LAYERS
print("Trainable scalars:", model.count_params())

## 11. Layer-weighted supervised loss
For every sample, sum the squared symbol-vector error at each layer, with logarithmic layer weights:
$$\mathcal L=\sum_i\sum_{t=1}^{T}\log(t+1)\|s_i-s_i^{(t)}\|_2^2.$$
The weights emphasize deeper outputs while supervising earlier layers. Tensor broadcasting repeats the target over the layer axis without allocating a separate tiled target. The summed reduction preserves the source notebook's objective; its magnitude depends on batch size and stream count.

In [ ]:
def layer_weighted_loss(outputs, targets):
    per_layer_error = tf.reduce_sum(tf.square(outputs - targets), axis=1)
    weights = tf.math.log(tf.cast(tf.range(2, tf.shape(outputs)[2] + 2), outputs.dtype))
    return tf.reduce_sum(per_layer_error * weights)

print("Initial loss:", float(layer_weighted_loss(model(train_inputs)[0], train_targets)))

## 12. Train with Adam and GradientTape
Each step performs one forward pass on the full training set, differentiates the weighted loss through all triangular solves and activations, and applies Adam to the layer scalars. `apply_gradients` is compatible with current Keras optimizers and replaces the removed `optimizer.minimize` API.

The compiled `train_step` reduces Python overhead. Non-finite losses/gradients stop execution rather than reporting an invalid result. Loss is logged every 100 steps and retained for the next figure. This notebook trains one model across all listed SNRs.

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

@tf.function
def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        outputs, _ = model(inputs)
        loss = layer_weighted_loss(outputs, targets)
        tf.debugging.assert_all_finite(loss, "Non-finite training loss")
    gradients = tape.gradient(loss, model.trainable_weights)
    for gradient in gradients:
        tf.debugging.assert_all_finite(gradient, "Non-finite gradient")
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))
    return loss

loss_history = []
for step in range(TRAIN_STEPS):
    loss_value = float(train_step(train_inputs, train_targets))
    loss_history.append(loss_value)
    if step == 0 or (step + 1) % 100 == 0 or step + 1 == TRAIN_STEPS:
        print(f"Step {step + 1:4d}/{TRAIN_STEPS}: loss = {loss_value:.6g}")

## 13. Inspect optimization and learned coefficients
The curve shows the training objective before each optimizer update. A lower training objective alone does not guarantee a lower test BER. The printed rows list the final `(alpha, beta, gamma)` for each layer and make the learned 45-scalar detector inspectable.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(loss_history) + 1), loss_history, color="tab:blue")
ax.set(xlabel="Optimizer step", ylabel="Layer-weighted training loss",
       title="TORNET training")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

learned_coefficients = np.array([[float(layer.alpha.numpy()), float(layer.beta.numpy()),
                                 float(layer.gamma.numpy())] for layer in model.tor_layers])
print("Layer    alpha       beta       gamma")
for k, row in enumerate(learned_coefficients, 1):
    print(f"{k:5d} {row[0]:10.5f} {row[1]:10.5f} {row[2]:10.5f}")

## 14. Inference and Gray-coded 16-QAM BER
Run TORNET on the test input list and use the last layer. For each real component, choose the nearest level in `[-3,-1,1,3]`; assign Gray labels `00,01,11,10`. This corrects the original notebook's natural binary lookup so the metric is explicitly Gray-coded BER.

Each complex 16-QAM symbol carries four bits. For each SNR, divide the number of mismatched bits by the actual number of tested bits. Explicit SNR labels avoid relying on equal block sizes. Zero observed errors are reported as zero; they do not establish that the true BER is zero.

In [ ]:
_, estimates = model(test_inputs, training=False)
s_hat = estimates.numpy()
assert np.isfinite(s_hat).all()

levels = np.array([-3, -1, 1, 3])
gray_bits = np.array([[0, 0], [0, 1], [1, 1], [1, 0]], dtype=np.int8)

def demodulate_bits(values):
    scaled = values[..., 0] * np.sqrt(ENERGY)
    level_indices = np.argmin(np.abs(scaled[..., None] - levels), axis=-1)
    return gray_bits[level_indices]

predicted_bits = demodulate_bits(s_hat)
true_bits = demodulate_bits(test_targets)
bit_errors = predicted_bits != true_bits
ber, error_counts, bit_counts = [], [], []
print("SNR (dB)    bit errors     tested bits       BER")
for snr in SNR_DB:
    errors = bit_errors[snr_labels[test_idx] == snr]
    count, total = int(errors.sum()), errors.size
    error_counts.append(count)
    bit_counts.append(total)
    ber.append(count / total)
    print(f"{snr:8.0f} {count:13d} {total:15d} {count/total:12.6g}")

## 15. TORNET BER curve
Plot the measured test BER against the configured SNR. Only TORNET is shown. No stored comparison numbers enter this figure. Any zero-error points are omitted from the logarithmic curve and identified below it; the preceding table preserves their exact counts.

In [ ]:
ber = np.asarray(ber)
fig, ax = plt.subplots()
ax.semilogy(SNR_DB, np.where(ber > 0, ber, np.nan), "o-", label="TORNET")
ax.set(xlabel="SNR (dB)", ylabel="Gray-coded bit error rate",
       title=f"TORNET direct LEO: {N_LAYERS} layers, {Nr} receive / {Nt} transmit, 16-QAM")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
if np.any(ber == 0):
    print("Zero observed errors at SNRs:", SNR_DB[ber == 0].tolist())

## 16. Reading and rerunning this experiment
- `TRAIN_STEPS` defaults to 2,000 full-batch optimizer steps. `TORNET_STEPS` overrides it for verification.
- `Nt`, `Nr`, and the direct-channel fading parameters define the channel distribution. Retrain after changing them.
- `data.H_direct[cc]` contains the normalized direct channel for channel seed `cc`.
- `N_LAYERS` controls detector depth; `BAND_WIDTH` controls the lower-band split.
- The alternating train/test split shares channel realizations. It measures performance on new symbols under seen channels; use disjoint seeds to test unseen-channel generalization.
- The regularizer retains the source experiment's simulated-noise variance convention, explained in Section 6.
- Only the TORNET detector is implemented. Orbital model selection, transfer, and energy studies are outside this experiment.

**Verification:** all code cells were executed sequentially with five training steps, including gradients, direct-channel identity checks, inference, and BER. The delivered notebook has cleared outputs and retains the 2,000-step default. Full training and notebook-viewer inspection were not performed.